# 04 — Short-answer grading: measuring the rules, fixing the thresholds

The Quizzes page can grade a typed answer with four rules (exact, near-string,
distractor, `AnswerScorer` ranking) and, for the ambiguous band, an LLM tie-break.
This notebook measures the rules on SciQ's **test split** (never seen by the scorer)
and picks `T_STRING` / `T_RATIO` by a stated criterion: **precision of "correct"
≥ 0.90 both with and without a support passage, then best F1**. A false "correct" teaches a wrong fact; a false "near"
costs one click on *I was right*.

What this is not: real typed answers. Positives are synthetic variants of the
reference; the override log in the app is where real labels come from.

In [1]:
from ml_pipeline import notebook_setup; notebook_setup.bootstrap()
import json, re, time, itertools
import numpy as np, pandas as pd
from ml_pipeline import config
from ml_pipeline.answer_scorer import AnswerScorer
from ml_pipeline.services import short_answer as sa

sciq = pd.read_parquet(config.PROCESSED_DIR / "sciq.parquet")
test = sciq[sciq.split == "test"].reset_index(drop=True)
scorer = AnswerScorer.load(config.MODELS_DIR / "answer_scorer.joblib")
len(test)

884

## Evaluation set

**Positives** — what a student who knows the answer might type: the reference
as-is; lowercased and unpunctuated; with / without a leading article; a plural
or singular; for multi-word answers, another word order and the answer minus
one function word. **Negatives** — the three distractors (plausible, on-topic,
wrong) and one answer from an unrelated question.

In [2]:
FUNCTION_WORDS = {"of", "the", "a", "an", "in", "and", "to"}
rng = np.random.default_rng(config.RANDOM_SEED)

def plural_variant(a):
    w = a.split()
    if len(w) != 1: return None
    return a[:-1] if a.endswith("s") and len(a) > 3 else a + "s"

def positives(ref):
    out = {ref, ref.lower(), re.sub(r"[^\w\s]", "", ref), "the " + ref}
    if ref.lower().startswith(("the ", "a ", "an ")):
        out.add(ref.split(" ", 1)[1])
    p = plural_variant(ref)
    if p: out.add(p)
    w = ref.split()
    if len(w) > 1:
        out.add(" ".join(reversed(w)))
        dropped = [x for x in w if x.lower() not in FUNCTION_WORDS]
        if len(dropped) < len(w): out.add(" ".join(dropped))
    return out

rows = []
answers_pool = test.correct_answer.tolist()
for i, r in test.iterrows():
    dist = [r.distractor1, r.distractor2, r.distractor3]
    for t in positives(r.correct_answer):
        rows.append(dict(qid=i, typed=t, label=1, kind="positive"))
    for d in dist:
        rows.append(dict(qid=i, typed=d, label=0, kind="distractor"))
    other = answers_pool[int(rng.integers(len(answers_pool)))]
    if other != r.correct_answer:
        rows.append(dict(qid=i, typed=other, label=0, kind="unrelated"))
ev = pd.DataFrame(rows)
ev.kind.value_counts()

kind
positive      2722
distractor    2652
unrelated      881
Name: count, dtype: int64

## Score everything once

The thresholds only gate numbers the rules compute; compute those once per
row (one batched `predict_proba` for the whole set) and sweep the thresholds
over the cached columns.

In [3]:
def features(ev, with_support=True):
    q = test.loc[ev.qid]
    ev = ev.copy()
    ev["ref"] = q.correct_answer.values
    ev["nt"] = ev.typed.map(sa.normalise); ev["nr"] = ev.ref.map(sa.normalise)
    ev["exact"] = ev.nt == ev.nr
    ev["sim"] = [sa.string_similarity(a, b) for a, b in zip(ev.nt, ev.nr)]
    ev["is_distractor"] = [nt in {sa.normalise(d) for d in (r.distractor1, r.distractor2, r.distractor3)}
                           for nt, r in zip(ev.nt, (test.loc[i] for i in ev.qid))]
    support = q.support.values if with_support else [""] * len(q)
    ev["has_support"] = [bool(str(s).strip()) for s in support]
    # one long frame: typed, reference, three distractors per row
    long = []
    for j, (qi, typed) in enumerate(zip(ev.qid, ev.typed)):
        r = test.loc[qi]
        for c in (typed, r.correct_answer, r.distractor1, r.distractor2, r.distractor3):
            long.append(dict(row=j, question=r.question, candidate=c, support=support[j]))
    long = pd.DataFrame(long)
    long["p"] = scorer.predict_proba(long)
    ev["s_typed"] = long[long.index % 5 == 0].p.values
    ev["s_ref"] = long[long.index % 5 == 1].p.values
    ev["s_dmax"] = long[long.index % 5 >= 2].groupby("row").p.max().values
    return ev

t0 = time.time(); F = features(ev); F0 = features(ev, with_support=False)
print(f"{len(F)} rows scored twice in {time.time()-t0:.1f}s")

6255 rows scored twice in 3.6s


In [4]:
def cascade(F, T_STRING, T_RATIO):
    v = np.where(F.exact, "correct",
        np.where(F.sim >= T_STRING, "correct",
        np.where(F.is_distractor, "wrong",
        np.where(F.has_support & (F.s_typed >= F.s_dmax) & (F.s_typed >= T_RATIO * F.s_ref), "correct",
        np.where(F.s_typed >= F.s_dmax, "near", "wrong")))))
    how = np.where(F.exact, "exact", np.where(F.sim >= T_STRING, "string",
          np.where(F.is_distractor, "distractor", "scorer")))
    return v, how

def metrics(F, v):
    correct = v == "correct"; pos = (F.label == 1).values
    tp = (correct & pos).sum(); fp = (correct & ~pos).sum(); fn = (~correct & pos).sum()
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    return dict(precision=round(float(p), 4), recall=round(float(r), 4),
                f1=round(float(2*p*r/max(p+r, 1e-9)), 4),
                near_rate=round(float(((v == "near") & pos).sum() / max(pos.sum(), 1)), 4),
                near_rate_neg=round(float(((v == "near") & ~pos).sum() / max((~pos).sum(), 1)), 4))

grid = [(ts, tr) for ts in (0.5, 0.6, 0.7, 0.8, 0.9) for tr in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0)]
# Each pair is measured with support (the explanation) and without it: a
# question with a blank explanation is graded on the no-support numbers,
# so the bar has to hold there too.
sweep = pd.DataFrame([dict(T_STRING=ts, T_RATIO=tr,
                           **metrics(F, cascade(F, ts, tr)[0]),
                           **{f"{k}_nosup": v for k, v in metrics(F0, cascade(F0, ts, tr)[0]).items()})
                      for ts, tr in grid])
ok = sweep[(sweep.precision >= 0.90) & (sweep.precision_nosup >= 0.90)]
chosen = (ok if len(ok) else sweep).sort_values(["f1", "precision_nosup"], ascending=False).iloc[0]
sweep.sort_values("f1", ascending=False).head(8)[["T_STRING", "T_RATIO", "precision", "recall", "f1", "precision_nosup", "recall_nosup"]]

,T_STRING,T_RATIO,precision,recall,f1,precision_nosup,recall_nosup
19,0.8,0.6,0.9851,0.9938,0.9894,0.9936,0.9636
20,0.8,0.7,0.9854,0.9919,0.9886,0.9936,0.9636
18,0.8,0.5,0.9833,0.9938,0.9885,0.9936,0.9636
21,0.8,0.8,0.9857,0.9904,0.9881,0.9936,0.9636
22,0.8,0.9,0.9861,0.9875,0.9868,0.9936,0.9636
23,0.8,1.0,0.9864,0.9842,0.9853,0.9936,0.9636
13,0.7,0.6,0.9713,0.9941,0.9826,0.9791,0.9655
14,0.7,0.7,0.9716,0.9927,0.9820,0.9791,0.9655


## Per-rule breakdown, and without support

Each rule alone, then the cascade at the chosen thresholds; then the same
cascade with `support = ""` — real questions often have a thin explanation.

In [5]:
TS, TR = float(chosen.T_STRING), float(chosen.T_RATIO)
def rule_only(F, how_name):
    v, how = cascade(F, TS, TR)
    return metrics(F, np.where(how == how_name, v, "wrong"))
components = {"exact": rule_only(F, "exact"), "string": rule_only(F, "string"),
              "distractor": rule_only(F, "distractor"), "scorer": rule_only(F, "scorer"),
              "cascade": metrics(F, cascade(F, TS, TR)[0])}
no_support = metrics(F0, cascade(F0, TS, TR)[0])
pd.DataFrame({**components, "cascade_no_support": no_support}).T

,precision,recall,f1,near_rate,near_rate_neg
exact,1.0000,0.6506,0.7883,0.0000,0.0000
string,0.9804,0.3130,0.4745,0.0000,0.0000
distractor,0.0000,0.0000,0.0000,0.0000,0.0000
scorer,0.7736,0.0301,0.0580,0.0000,0.0433
cascade,0.9851,0.9938,0.9894,0.0000,0.0433
cascade_no_support,0.9936,0.9636,0.9784,0.0114,0.1019


## The LLM tie-break, if a provider is reachable

Only the `near` band would be sent to the LLM. If Ollama is serving locally,
ask it the one narrow question the app asks, on a sample of that band, and
report accuracy and median latency. Skipped (recorded as `null`) otherwise.

In [6]:
import urllib.request, urllib.error
def llm_same(question, reference, typed, model="llama3.2:3b", url="http://localhost:11434/api/generate"):
    prompt = (f"Reference answer: {json.dumps(reference)}. A student wrote: {json.dumps(typed)}. "
              f"Question: {json.dumps(question)}. Does the student's answer mean the same thing "
              "as the reference? Reply with JSON only: {\"same\": true} or {\"same\": false}.")
    body = json.dumps({"model": model, "prompt": prompt, "stream": False, "format": "json"}).encode()
    req = urllib.request.Request(url, data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        return bool(json.loads(json.loads(resp.read())["response"]).get("same"))

v, _ = cascade(F, TS, TR)
near = F[v == "near"]
llm = None
try:
    sample = near.sample(min(60, len(near)), random_state=config.RANDOM_SEED)
    t0 = time.time(); preds = []
    for _, r in sample.iterrows():
        preds.append(llm_same(test.loc[r.qid].question, r.ref, r.typed))
    lat = (time.time() - t0) / len(sample)
    acc = float(np.mean(np.array(preds) == (sample.label.values == 1)))
    llm = dict(n=int(len(sample)), accuracy=round(acc, 4), mean_latency_s=round(lat, 2),
               near_share=round(float((v == "near").mean()), 4))
except (urllib.error.URLError, OSError, KeyError, ValueError) as exc:
    print("LLM tier skipped:", exc)
llm

{'n': 60, 'accuracy': 1.0, 'mean_latency_s': 0.35, 'near_share': 0.0245}

## Result

In [7]:
report = dict(
    chosen=dict(T_STRING=TS, T_RATIO=TR, precision=float(chosen.precision),
                recall=float(chosen.recall), f1=float(chosen.f1), near_rate=float(chosen.near_rate)),
    components=components, no_support=no_support, llm=llm,
    n_positive=int((ev.label == 1).sum()), n_negative=int((ev.label == 0).sum()),
    split="test", criterion="precision >= 0.90 with and without support, then max F1",
    caveats=[
        "Positives are synthetic variants of the reference; most are settled by the exact "
        "and string rules, so the scorer and LLM tiers are exercised mainly by negatives.",
        "No synthetic positive landed in the near band, so the LLM accuracy above reflects "
        "only its ability to reject wrong answers; its ability to accept a true paraphrase is "
        "unmeasured here and is what the in-app override log will show.",
    ],
)
config.ensure_dirs()
(config.REPORTS_DIR / "short_answer_metrics.json").write_text(json.dumps(report, indent=2))
report["chosen"]

{'T_STRING': 0.8,
 'T_RATIO': 0.6,
 'precision': 0.9851,
 'recall': 0.9938,
 'f1': 0.9894,
 'near_rate': 0.0}